# 🌍 Earth Engine → NetCDF Exporter

**Generic, fault-tolerant pipeline** for downloading any Google Earth Engine
`ImageCollection` as **year-wise, CF-compliant NetCDF** files.

| Feature | Detail |
|---|---|
| **Any resolution** | 30 m CDL ↔ 300 km GRACE — auto-detected |
| **Any temporal frequency** | Daily, monthly, annual — aggregated per year |
| **CF-1.8 compliant** | Standard global attrs, coordinate metadata, clean encoding |
| **Resume on crash** | Skips years already exported to Drive |
| **Data integrity** | Pre-write validation + post-write re-read verification |
| **Compression** | zlib — dramatically smaller files |
| **Edge-case safe** | 1 image/year, SR-ORG CRS, complex polygons |

### Usage
1. Edit the **Configuration** cell (typically just `DATASET_ID`)
2. Upload / mount your shapefile
3. **Runtime → Run all** (`Ctrl+F9`)
4. Files appear in `Drive / MSUGWB / <dataset> /`

In [ ]:
# ╔════════════════════════════════════════════════════════════════╗
# ║  CONFIGURATION — edit this cell for each dataset run           ║
# ╚════════════════════════════════════════════════════════════════╝

# ── Dataset ──────────────────────────────────────────────────────
# Examples:
#   "USDA/NASS/CDL"                  30 m   annual
#   "GRIDMET/DROUGHT"                ~4 km  pentad
#   "MODIS/061/MCD12Q1"              500 m  annual
#   "MODIS/061/MOD13A3"              1 km   monthly
#   "NASA/GRACE/MASS_GRIDS/LAND"     ~300 km monthly
DATASET_ID = "USDA/NASS/CDL"
PROJECT_ID = "msugw-503806"

# ── Region of interest ───────────────────────────────────────────
SHAPEFILE_DIR = "/content/ogallala_shp"   # folder with .shp + sidecars
ROI_LABEL     = "Ogallala"                # used in filenames

# ── Output ───────────────────────────────────────────────────────
DRIVE_ROOT = "/content/drive/MyDrive/MSUGWB"

# ── Year range (None = auto-detect from collection metadata) ─────
YEAR_START = None
YEAR_END   = None

# ── Processing knobs ─────────────────────────────────────────────
DASK_WORKERS    = 4      # parallel threads (keep <= 6 for GEE quotas)
CHUNK_XY        = 512    # dask spatial chunk size (pixels per dim)
COMPRESS_LEVEL  = 4      # zlib  0 = off · 9 = max
MAX_RETRIES     = 3      # per-year retries for transient errors
FORCE_OVERWRITE = False  # True → re-process files already on Drive

In [ ]:
# ═════════════════════════════════════════════════════════════════
#  Install · Import · Authenticate · Mount Drive
# ═════════════════════════════════════════════════════════════════

!pip install -q xee xarray netcdf4 geopandas pyproj

import os, sys, glob, time, shutil, logging, tempfile
from datetime import datetime, timezone

import numpy as np
import ee
import xarray as xr
import xee                       # registers the 'ee' xarray engine
from xee import helpers
import geopandas as gpd
import pyproj
import dask
import requests

# ── Logging ──
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)-7s | %(message)s',
    datefmt='%H:%M:%S',
)
log = logging.getLogger('gee_export')
logging.getLogger('urllib3.connectionpool').setLevel(logging.ERROR)

# ── HTTP pool (avoids dropped connections on long runs) ──
_adp = requests.adapters.HTTPAdapter(pool_connections=50, pool_maxsize=50)
_ses = requests.Session()
_ses.mount('https://', _adp)

# ── Dask ──
dask.config.set(scheduler='threads', num_workers=DASK_WORKERS)

# ── Google Drive ──
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    log.info('Not on Colab -- Drive mount skipped')

# ── Earth Engine ──
ee.Authenticate()
ee.Initialize(project=PROJECT_ID)
log.info('Earth Engine ready  (project=%s)', PROJECT_ID)

In [ ]:
# ═════════════════════════════════════════════════════════════════
#  Pipeline Functions
# ═════════════════════════════════════════════════════════════════

# ─── ROI ─────────────────────────────────────────────────────────

def load_roi(shapefile_dir):
    '''Load shapefile -> dissolved EPSG:4326 geometry.
    Auto-simplifies when vertex count exceeds Earth Engine limits.
    '''
    paths = glob.glob(os.path.join(shapefile_dir, '**', '*.shp'), recursive=True)
    if not paths:
        raise FileNotFoundError(f'No .shp found in {shapefile_dir}')
    log.info('Shapefile: %s', paths[0])
    gdf = gpd.read_file(paths[0])
    if gdf.crs is None or gdf.crs.to_epsg() != 4326:
        gdf = gdf.to_crs(epsg=4326)

    # Dissolve — compatible across geopandas versions
    roi = (gdf.geometry.union_all()
           if hasattr(gdf.geometry, 'union_all')
           else gdf.geometry.unary_union)

    # Count vertices
    try:
        import shapely as _shp
        nv = int(_shp.get_num_coordinates(roi))
    except Exception:
        nv = len(roi.wkt) // 20

    if nv > 50_000:
        roi = roi.simplify(0.001, preserve_topology=True)
        log.warning('Simplified ROI from %s vertices (tol=0.001 deg)', f'{nv:,}')
    else:
        log.info('ROI vertices: %s', f'{nv:,}')
    return roi


# ─── CRS & Scale ────────────────────────────────────────────────

def resolve_crs_and_scale(collection):
    '''Auto-detect native CRS + pixel scale. Always returns a standard
    EPSG code and scale in that CRS's units.

    Why this is critical
    --------------------
    * Custom WKT CRS strings (e.g. ERDAS "IMAGINE GeoTIFF") are parseable
      by pyproj but cause XEE to produce a mis-aligned grid, returning
      all-NaN data.
    * SR-ORG codes (e.g. MODIS Sinusoidal SR-ORG:6974) cannot be parsed
      by pyproj at all.
    This function resolves both cases to a well-known EPSG code.
    '''
    proj = collection.first().select(0).projection()
    info = proj.getInfo()
    native_crs = info.get('crs', 'EPSG:4326')
    nominal_m  = proj.nominalScale().getInfo()

    log.info('Native CRS    : %s', native_crs[:70])
    log.info('Nominal scale : %.1f m', nominal_m)

    resolved = 'EPSG:4326'   # ultimate fallback

    if native_crs.startswith('SR-ORG:'):
        log.warning('SR-ORG CRS (%s) -> EPSG:4326 fallback', native_crs)
    elif native_crs.startswith('EPSG:'):
        resolved = native_crs
        log.info('Already EPSG: %s', resolved)
    else:
        try:
            crs_obj = pyproj.CRS.from_user_input(native_crs)
            epsg = crs_obj.to_epsg()
            if epsg:
                resolved = f'EPSG:{epsg}'
                log.info('Resolved to EPSG:%d', epsg)
            elif crs_obj.axis_info and crs_obj.axis_info[0].unit_name == 'metre':
                resolved = 'EPSG:5070'
                log.warning('Metre-based CRS, no EPSG -> EPSG:5070 (CONUS Albers)')
            else:
                log.warning('No EPSG match -> EPSG:4326')
        except Exception as exc:
            log.warning('CRS parse error: %s -> EPSG:4326', exc)

    # Compute scale in resolved CRS units
    try:
        unit = pyproj.CRS.from_user_input(resolved).axis_info[0].unit_name
    except Exception:
        unit = 'degree'

    if unit == 'metre':
        scale = round(nominal_m, 2)
    else:
        scale = nominal_m / 111_320.0   # metres -> approx degrees

    log.info('Output CRS    : %s  |  scale = %.6g %s', resolved, scale, unit)
    return resolved, scale


# ─── Encoding ───────────────────────────────────────────────────

def strip_xee_encoding(ds):
    '''Remove ALL encoding injected by the XEE engine.

    CRITICAL FIX: XEE leaks the CRS-transform pixel size (e.g. 30 m)
    into the variable encoding as ``scale_factor: 30.0``.  xarray's
    ``to_netcdf()`` writes this as a CF packing attribute.  On read-back
    every value is multiplied by 30, destroying all data.
    '''
    for name in list(ds.data_vars) + list(ds.coords):
        if name in ds:
            ds[name].encoding.clear()
    return ds


def build_encoding(ds, compress_level=4):
    '''Explicit float32 + NaN fill + zlib for data vars; float64 for coords.'''
    enc = {}
    for var in ds.data_vars:
        enc[var] = {
            'dtype': 'float32',
            '_FillValue': np.float32(np.nan),
            'zlib': compress_level > 0,
            'complevel': compress_level,
        }
    for c in ds.coords:
        if c == 'time':
            enc[c] = {'dtype': 'float64', '_FillValue': None}
        elif c in ('x', 'y'):
            enc[c] = {'dtype': 'float64', '_FillValue': None}
    return enc


# ─── CF Compliance ──────────────────────────────────────────────

def make_cf_compliant(ds, dataset_id, roi_label, grid_crs, year):
    '''Stamp CF-1.8 global + coordinate attributes.'''
    ds.attrs.update({
        'Conventions': 'CF-1.8',
        'title':  f'{dataset_id} -- {roi_label} ({year})',
        'source': f'Google Earth Engine: {dataset_id}',
        'history': f'Created {datetime.now(timezone.utc).isoformat()}',
        'crs': grid_crs,
    })

    if 'time' in ds.coords:
        ds['time'].attrs.update(axis='T', standard_name='time')

    try:
        is_proj = (pyproj.CRS.from_user_input(grid_crs)
                   .axis_info[0].unit_name == 'metre')
    except Exception:
        is_proj = False

    if 'x' in ds.coords:
        ds['x'].attrs.update(
            axis='X',
            standard_name='projection_x_coordinate' if is_proj else 'longitude',
            units='m' if is_proj else 'degrees_east',
        )
    if 'y' in ds.coords:
        ds['y'].attrs.update(
            axis='Y',
            standard_name='projection_y_coordinate' if is_proj else 'latitude',
            units='m' if is_proj else 'degrees_north',
        )
    return ds


# ─── Pre-Write Validation ──────────────────────────────────────

def pre_write_check(ds, year):
    '''Load a small centre sample and raise RuntimeError if all NaN.'''
    var0 = list(ds.data_vars)[0]
    da = ds[var0]
    yd = next((d for d in ('y', 'latitude', 'lat') if d in da.dims), None)
    xd = next((d for d in ('x', 'longitude', 'lon') if d in da.dims), None)
    if not yd or not xd:
        log.warning('[%d] Cannot find spatial dims -- skipping pre-check', year)
        return

    ny, nx = da.sizes[yd], da.sizes[xd]
    w = min(100, ny // 4, nx // 4, 200)
    if w < 1:
        w = 1
    yc, xc = ny // 2, nx // 2

    idx = {}
    if 'time' in da.dims:
        idx['time'] = 0
    idx[yd] = slice(max(0, yc - w), min(ny, yc + w))
    idx[xd] = slice(max(0, xc - w), min(nx, xc + w))

    sample = da.isel(**idx).compute().values
    nf = int(np.isfinite(sample).sum())
    log.info('[%d] Pre-write: %s / %s finite in centre', year,
             f'{nf:,}', f'{sample.size:,}')

    if nf == 0:
        raise RuntimeError(
            f'[{year}] ALL-NaN -- Earth Engine returned no valid pixels. '
            'Verify CRS, geometry, and year availability.'
        )
    pct = 100.0 * nf / sample.size
    if pct < 1.0:
        log.warning('[%d] Only %.1f%% finite -- possible partial coverage', year, pct)


# ─── Post-Write Integrity Verification ─────────────────────────

def post_write_verify(filepath, expected_vars, year):
    '''Reopen the NetCDF from disk and verify:
      1. All bands present
      2. Time dim exists
      3. Coords finite
      4. No bogus scale_factor
      5. Data has finite values
    Returns (ok, message).
    '''
    try:
        vds = xr.open_dataset(filepath, chunks='auto')
    except Exception as e:
        return False, f'Cannot reopen: {e}'

    try:
        # All expected variables?
        missing = [v for v in expected_vars if v not in vds.data_vars]
        if missing:
            return False, f'Missing: {missing}'

        # Time dimension?
        if 'time' not in vds.dims:
            return False, 'No time dimension'

        # Coordinates finite?
        for c in vds.coords:
            v = vds[c].values
            if np.issubdtype(v.dtype, np.floating):
                if not np.all(np.isfinite(v)):
                    return False, f'Non-finite coord: {c}'

        # No bogus scale_factor?
        for v in expected_vars:
            sf = vds[v].encoding.get('scale_factor')
            if sf is not None and sf != 1.0:
                return False, f'scale_factor={sf} on {v}'

        # Data has finite values in centre?
        da = vds[expected_vars[0]]
        idx = {}
        if 'time' in da.dims:
            idx['time'] = 0
        for d in da.dims:
            if d != 'time':
                s = da.sizes[d]
                m = s // 2
                hw = min(50, s // 4, 100)
                if hw < 1:
                    hw = 1
                idx[d] = slice(max(0, m - hw), min(s, m + hw))
        samp = da.isel(**idx).compute().values
        nf = int(np.isfinite(samp).sum())
        if nf == 0:
            return False, 'All NaN on re-read'
        pct = 100.0 * nf / samp.size
        return True, f'{pct:.1f}% finite in verify sample'

    finally:
        vds.close()

## 🚀 Pipeline Execution

The cell below processes **all years** sequentially:

1. **Resume check** — skips years whose `.nc` already exists on Drive
   (also detects corrupt partial files < 1 KiB and re-downloads them)
2. **Open via XEE** — lazy dask-backed dataset, no eager download
3. **Strip encoding** — removes the bogus `scale_factor` from CRS metadata
4. **CF compliance** — stamps standard global + coordinate attributes
5. **Pre-write validation** — loads a centre sample, aborts if all NaN
6. **Write to scratch** — `/content/` (fast local SSD) with zlib compression
7. **Post-write verification** — re-opens the file from disk, checks integrity
8. **Export to Drive** — only after verification passes

> **After a crash:** just *Run All* again — completed years are auto-skipped.

In [ ]:
# ═════════════════════════════════════════════════════════════════
#  Run Pipeline
# ═════════════════════════════════════════════════════════════════

log.info('=' * 62)
log.info('  Dataset : %s', DATASET_ID)
log.info('=' * 62)

# ── 1. Load ROI ─────────────────────────────────────────────────
roi = load_roi(SHAPEFILE_DIR)

# ── 2. Output folder ────────────────────────────────────────────
short = DATASET_ID.split('/')[-1]
out_dir = os.path.join(DRIVE_ROOT, short)
os.makedirs(out_dir, exist_ok=True)
log.info('Output folder : %s', out_dir)

# ── 3. Collection metadata ──────────────────────────────────────
coll = ee.ImageCollection(DATASET_ID)

bands = coll.first().bandNames().getInfo()
if not bands:
    raise ValueError(f'{DATASET_ID} returned no bands')

try:
    y0 = YEAR_START or int(ee.Date(
        coll.sort('system:time_start').first()
            .get('system:time_start')).get('year').getInfo())
    y1 = YEAR_END or int(ee.Date(
        coll.sort('system:time_start', False).first()
            .get('system:time_start')).get('year').getInfo())
except Exception as exc:
    raise ValueError(
        f'Cannot determine year range for {DATASET_ID}: {exc}'
    ) from exc

years = list(range(y0, y1 + 1))
log.info('Bands (%d) : %s', len(bands), bands)
log.info('Years      : %d - %d  (%d total)', y0, y1, len(years))

# ── 4. CRS & grid ───────────────────────────────────────────────
grid_crs, px = resolve_crs_and_scale(coll.select(bands))

grid_params = helpers.fit_geometry(
    geometry=roi,
    geometry_crs='EPSG:4326',
    grid_crs=grid_crs,
    grid_scale=(px, -px),
)
log.info('Grid shape : %s', grid_params.get('shape_2d', '?'))

# ── 5. Year-by-year export ──────────────────────────────────────
n_ok = n_fail = n_skip = 0

for year in years:
    fname = f'{short}_{year}_{ROI_LABEL}.nc'
    final = os.path.join(out_dir, fname)

    # ── Resume: skip if file exists on Drive ──
    if os.path.exists(final) and not FORCE_OVERWRITE:
        sz = os.path.getsize(final)
        if sz < 1024:
            # Likely a corrupt partial file from a previous crash
            log.warning('[%d] Tiny file on Drive (%d B) -- re-processing', year, sz)
            os.remove(final)
        else:
            log.info('[%d] On Drive (%.1f MiB) -- skip', year, sz / 1024**2)
            n_skip += 1
            continue

    tmp = os.path.join('/content', fname)
    done = False

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            log.info('[%d] attempt %d/%d', year, attempt, MAX_RETRIES)

            # ── Filter collection to this year ──
            yr_col = (
                ee.ImageCollection(DATASET_ID)
                .filter(ee.Filter.calendarRange(year, year, 'year'))
                .select(bands)
                .sort('system:time_start')
            )

            n_img = yr_col.size().getInfo()
            if n_img == 0:
                log.warning('[%d] 0 images -- skip year', year)
                break   # not a failure, just no data this year

            log.info('[%d] %d image(s) in collection', year, n_img)

            # ── Open lazily via XEE ──
            ds = xr.open_dataset(
                yr_col,
                engine='ee',
                chunks={'x': CHUNK_XY, 'y': CHUNK_XY},
                **grid_params,
            )

            # ── Fix encoding ──
            ds = strip_xee_encoding(ds)

            # ── CF attributes ──
            ds = make_cf_compliant(ds, DATASET_ID, ROI_LABEL, grid_crs, year)

            # ── Validate before expensive write ──
            pre_write_check(ds, year)

            # ── Write to Colab scratch ──
            enc = build_encoding(ds, COMPRESS_LEVEL)
            log.info('[%d] Writing -> %s', year, tmp)
            t0 = time.time()
            ds.to_netcdf(tmp, engine='netcdf4', encoding=enc)
            dt = time.time() - t0
            mb = os.path.getsize(tmp) / 1024**2
            log.info('[%d] Written in %.0fs  (%.1f MiB)', year, dt, mb)

            if os.path.getsize(tmp) < 1024:
                raise RuntimeError(
                    f'File suspiciously small ({os.path.getsize(tmp)} B)')

            # ── Post-write integrity check ──
            ok_flag, msg = post_write_verify(tmp, bands, year)
            if not ok_flag:
                raise RuntimeError(f'INTEGRITY FAIL: {msg}')
            log.info('[%d] Integrity OK: %s', year, msg)

            # ── Export to Google Drive ──
            log.info('[%d] Exporting to Drive ...', year)
            t_mv = time.time()
            shutil.copy2(tmp, final)       # copy first (cross-fs)
            os.remove(tmp)                 # then remove temp
            log.info('[%d] Exported in %.0fs -> %s',
                     year, time.time() - t_mv, final)

            n_ok += 1
            done = True
            break

        except RuntimeError as e:
            log.error('[%d] %s', year, e)
            n_fail += 1
            break   # non-retryable

        except Exception as e:
            log.error('[%d] %s', year, e)
            if attempt < MAX_RETRIES:
                wait = 30 * 2 ** (attempt - 1)
                log.info('[%d] retry in %ds ...', year, wait)
                time.sleep(wait)
            else:
                log.error('[%d] retries exhausted', year)
                n_fail += 1

        finally:
            # Clean up scratch on failure
            if not done and os.path.exists(tmp):
                try:
                    os.remove(tmp)
                except OSError:
                    pass

# ── Summary ──────────────────────────────────────────────────────
log.info('=' * 62)
log.info('  DONE  %d ok | %d fail | %d skip  (%d years)',
         n_ok, n_fail, n_skip, len(years))
log.info('=' * 62)

# ── List files on Drive ──
if os.path.isdir(out_dir):
    nc_files = sorted(f for f in os.listdir(out_dir) if f.endswith('.nc'))
    if nc_files:
        log.info('Files on Drive (%s):', out_dir)
        for f in nc_files:
            sz = os.path.getsize(os.path.join(out_dir, f)) / 1024**2
            log.info('  %s  (%.1f MiB)', f, sz)